In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt

PATH = "feature_engineered/feature_engineered_CMG_RRV.csv"
df = pd.read_csv(PATH, parse_dates=['date'])
df = df.sort_values('date')
df.reset_index(drop=True, inplace=True)

In [ ]:
len(df.columns)

In [ ]:
df.columns.tolist()

In [ ]:
all_features = [
#  'date',
 'close',
 'open',
 'high',
 'low',
 'volume',
 'change',
#  'symbol',
 'return_day',
 'return_week',
 'return_month',
 'volatility_day',
 'volatility_week',
 'volatility_month',
 'liquidity_day',
 'liquidity_week',
 'liquidity_month',
 'high_minus_close',
 'low_minus_open',
 'cumulative_return',
 'stochastic_osc',
 'atr',
 'sma_3',
 'sma_7',
 'sma_14',
 'sma_21',
 'sma_50',
 'sma_100',
 'wma_3',
 'wma_7',
 'wma_14',
 'wma_21',
 'wma_50',
 'wma_100',
 'ema_6',
 'ema_12',
 'ema_26',
 'out_macd',
 'out_macd_signal',
 'out_macd_hist',
 'rsi_6',
 'rsi_12',
 'rsi_14',
 'stochrsi_6',
 'stochrsi_12',
 'stochrsi_14',
 'bbands_middle',
 'bbands_upper',
 'bbands_lower',
 'obv',
 'mfi_14',
 'mom_1',
 'mom_3',
 'mom_7',
 'cci_12',
 'cci_20',
 'rocr_3',
 'rocr_12',
 'willr',
 'trix',
 'net_profit_loss_before_tax',
 'depreciation_fixed_assets',
 'credit_risk_reserve',
 'unrealized_forex_gain_loss',
 'investment_income_loss',
 'interest_income',
 'interest_dividend_income',
 'net_cash_flow_operating_before_wc_changes',
 'change_in_receivables',
 'change_in_inventories',
 'change_in_payables',
 'change_in_prepaid_expenses',
 'interest_expense_paid',
 'corporate_income_tax_paid',
 'other_cash_inflows_operating',
 'other_cash_outflows_operating',
 'net_cash_flow_from_business_activities',
 'capital_expenditure',
 'cash_from_fixed_assets_sale',
 'cash_outflow_loans_debt_instruments',
 'cash_inflow_loan_recoveries',
 'investments_in_other_businesses',
 'cash_from_sale_investments',
 'dividends_profit_received',
 'cash_flow_investing',
 'increase_in_equity',
 'cash_used_share_repurchase',
 'cash_from_borrowings',
 'cash_repayment_borrowings',
 'dividends_paid',
 'cash_flow_financing',
 'net_cash_flow_period',
 'cash_and_equivalents',
 'exchange_rate_effect',
 'ending_cash_equivalents',
 'revenue_growth_percent',
 'revenue_vnd',
 'net_profit_after_tax_parent_vnd',
 'profit_growth_percent',
 'financial_income',
 'interest_expense',
 'sales_revenue_service_income',
 'revenue_deductions',
 'net_revenue',
 'cogs',
 'gross_profit',
 'financial_expenses',
 'profit_loss_associates',
 'selling_expenses',
 'administrative_expenses',
 'operating_profit_loss',
 'other_income',
 'profit_loss_joint_ventures',
 'other_income_expense',
 'other_profit',
 'profit_before_tax',
 'current_corporate_tax_expense',
 'deferred_corporate_tax_expense',
 'net_profit',
 'minority_interest',
 'parent_company_shareholders',
 'loans_to_equity_ratio',
 'debt_to_equity',
 'fixed_assets_to_equity',
 'equity_to_chartered_capital',
 'asset_turnover',
 'fixed_asset_turnover',
 'avg_collection_period',
 'avg_inventory_days',
 'avg_payment_days',
 'cash_cycle',
 'inventory_turnover',
 'ebit_margin_percent',
 'gross_profit_margin_percent',
 'net_profit_margin_percent',
 'roe_percent',
 'roic_percent',
 'roa_percent',
 'ebitda_billion_vnd',
 'ebit_billion_vnd',
 'dividend_yield_percent',
 'current_ratio',
 'cash_ratio',
 'quick_ratio',
 'interest_coverage',
 'financial_leverage',
 'market_cap_billion_vnd',
 'shares_outstanding_million',
 'pe',
 'pb',
 'ps',
 'p_cash_flow',
 'eps_vnd',
 'bvps_vnd',
 'ev_ebitda',
]

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

In [ ]:
df.head()

# Feature Selection

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression, f_regression
import seaborn as sns
import matplotlib.pyplot as plt

target = 'close'  

X = df.select_dtypes(include=[np.number]).drop(columns=[target])
y = df[target]

mi_scores = mutual_info_regression(X, y)
mi_series = pd.Series(mi_scores, index=X.columns)

_, p_values = f_regression(X, y)
p_series = pd.Series(p_values, index=X.columns)

selected_features = mi_series[(p_series < 0.05)].sort_values(ascending=False)

# 4. Draw correlation matrix of selected features
top_features = selected_features.head(15).index  # You can adjust this
corr_matrix = df[top_features].corr()

# Plot correlation matrix
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix of Selected Features")
plt.show()

# Scaler

Standardization and normalization

In [ ]:
# Lọc ra những cột có trong dữ liệu
feature_cols = [col for col in all_features if col in df.columns]
df = df[['date'] + feature_cols].dropna().reset_index(drop=True)

In [ ]:
# 2. Chuẩn hóa dữ liệu cho các feature (không chuẩn hóa cột 'date')
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(df[feature_cols])
df_scaled = pd.DataFrame(scaled_features, columns=feature_cols)
df_scaled['date'] = df['date']

# 3. Chuyển cột 'date' về kiểu datetime và đặt làm index để dễ group theo tháng
df_scaled['date'] = pd.to_datetime(df_scaled['date'])
df_scaled.set_index('date', inplace=True)

In [ ]:
def create_monthly_sequences(dataframe, label_col='close'):
    X_list = []
    y_list = []
    month_list = []
    
    # Group theo tháng (sử dụng pd.Grouper với freq='M' để lấy ngày cuối tháng)
    monthly_groups = list(dataframe.groupby(pd.Grouper(freq='M')))
    # Duyệt qua các tháng, ngoại trừ tháng cuối cùng (không có tháng tiếp theo)
    for i in range(len(monthly_groups) - 1):
        month_end, month_data = monthly_groups[i]
        next_month_end, next_month_data = monthly_groups[i+1]
        
        # Input: toàn bộ dữ liệu của tháng hiện tại (mảng có shape: [số ngày, n_features])
        X_seq = month_data.values
        
        # Lấy giá đóng cửa của ngày cuối tháng hiện tại và tháng tiếp theo
        current_close = month_data.iloc[-1][label_col]
        next_close = next_month_data.iloc[-1][label_col]
        
        # Tính phần trăm biến động
        pct_change = (next_close - current_close) / current_close
        # Đảm bảo label nằm trong khoảng [-1, 1]
        pct_change = np.clip(pct_change, -1, 1)
        
        X_list.append(X_seq)
        y_list.append(pct_change)
        month_list.append(month_end)
    
    return X_list, y_list, month_list

X_monthly, y_monthly, month_dates = create_monthly_sequences(df_scaled.copy(), label_col='close')

In [ ]:
print("Số mẫu (tháng):", len(X_monthly))
print("Ví dụ label:", y_monthly[:5])

In [ ]:
# 5. Vì mỗi tháng có số ngày giao dịch khác nhau, ta cần pad (đệm) các sequence về cùng một độ dài.
max_len = max([x.shape[0] for x in X_monthly])
n_features = len(feature_cols)

def pad_sequence(seq, target_length, n_features):
    if seq.shape[0] < target_length:
        pad = np.zeros((target_length - seq.shape[0], n_features))
        return np.vstack([seq, pad])
    return seq

X_monthly_padded = np.array([pad_sequence(x, max_len, n_features) for x in X_monthly])
y_monthly = np.array(y_monthly)

In [ ]:
print("X_monthly_padded shape:", X_monthly_padded.shape)  # (n_samples, max_len, n_features)

In [ ]:
# 6. Chia dữ liệu thành train/test (ví dụ 80% train)
n_samples = X_monthly_padded.shape[0]
train_size = int(n_samples * 0.8)

X_train = X_monthly_padded[:train_size]
y_train = y_monthly[:train_size]
X_test  = X_monthly_padded[train_size:]
y_test  = y_monthly[train_size:]

print("Train size:", X_train.shape, y_train.shape)
print("Test size:", X_test.shape, y_test.shape)

In [ ]:
# 7. Xây dựng mô hình LSTM với các kỹ thuật regularization
model = Sequential()
model.add(LSTM(64, return_sequences=True, input_shape=(max_len, n_features), kernel_regularizer=l2(1e-4)))
model.add(Dropout(0.3))
model.add(LSTM(64, kernel_regularizer=l2(1e-4)))
model.add(Dropout(0.3))
# Sử dụng activation 'tanh' để đầu ra nằm trong khoảng [-1, 1]
model.add(Dense(1, activation='tanh'))

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

In [ ]:
# 8. Thiết lập callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
checkpoint = ModelCheckpoint('best_lstm_monthly_model.h5', monitor='val_loss', save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

# 9. Huấn luyện mô hình
history = model.fit(X_train, y_train, epochs=100, batch_size=8, validation_split=0.1,
                    callbacks=[early_stop, checkpoint, reduce_lr], verbose=1)

With 153 features, you may have redundant or noisy information. It can be beneficial to first explore feature selection or dimensionality reduction (e.g., PCA) rather than simply increasing model complexity.

In [ ]:
# 10. Dự đoán và vẽ biểu đồ
y_pred = model.predict(X_test)
plt.figure(figsize=(10,5))
plt.plot(y_test, label='Label thực tế')
plt.plot(y_pred, label='Dự đoán')
plt.title('Phần trăm biến động (trend) thực tế vs dự đoán theo tháng')
plt.xlabel('Mẫu (theo thứ tự các tháng trong tập test)')
plt.ylabel('% Biến động')
plt.legend()
plt.show()